# Declarative Sharing (Application Packages TYPE=DATA)

Modern, versioned data sharing using **application packages**. Instead of raw `CREATE SHARE`,
you wrap your data in a **versioned, manifest-driven package** with release channels and
controlled rollouts.

**Key concepts:**
- **Application Package (TYPE=DATA)**: A container that bundles shared data, metadata, and optional notebooks into a single distributable unit.
- **Manifest (`manifest.yml`)**: Declares which databases, schemas, tables, views, and roles are shared — no setup script needed.
- **LIVE version**: A mutable development workspace for iterating before committing an immutable release.
- **BUILD → COMMIT → RELEASE**: The lifecycle for publishing changes to consumers.
- **Shared-by-reference vs shared-by-copy**: Tables/views are shared by reference; functions/procedures/agents are shared by copy (and must live in separate schemas).

This notebook demonstrates both the **new declarative approach** (`TYPE=DATA` with manifest) and the **legacy versioned approach** (`ADD VERSION` with setup scripts) for comparison.

## 1. Setup

In [ ]:
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE DATABASE decl_sharing_demo;
CREATE OR REPLACE SCHEMA decl_sharing_demo.curated_data;
CREATE OR REPLACE SCHEMA decl_sharing_demo.app_stage;
CREATE STAGE IF NOT EXISTS decl_sharing_demo.app_stage.v1_files;

USE ROLE SECURITYADMIN;
CREATE ROLE IF NOT EXISTS share_admin;
CREATE ROLE IF NOT EXISTS share_monitor;
GRANT ROLE share_admin TO ROLE SYSADMIN;
GRANT ROLE share_monitor TO ROLE share_admin;
GRANT CREATE SHARE ON ACCOUNT TO ROLE share_admin;
GRANT MANAGE SHARE TARGET ON ACCOUNT TO ROLE share_admin;
GRANT USAGE ON DATABASE decl_sharing_demo TO ROLE share_admin;
GRANT USAGE ON ALL SCHEMAS IN DATABASE decl_sharing_demo TO ROLE share_admin;
GRANT CREATE VIEW ON SCHEMA decl_sharing_demo.curated_data TO ROLE share_admin;
GRANT IMPORTED PRIVILEGES ON DATABASE snowflake TO ROLE share_monitor;

## 2. Create Sample Data

In [ ]:
CREATE OR REPLACE TABLE decl_sharing_demo.curated_data.orders AS
SELECT
    o_orderkey,
    o_custkey,
    o_orderstatus,
    o_totalprice,
    o_orderdate,
    o_orderpriority,
    o_clerk
FROM snowflake_sample_data.tpch_sf1.orders
LIMIT 5000;

CREATE OR REPLACE TABLE decl_sharing_demo.curated_data.customers AS
SELECT
    c_custkey,
    c_name,
    c_address,
    c_nationkey,
    c_phone,
    c_acctbal,
    c_mktsegment
FROM snowflake_sample_data.tpch_sf1.customer
LIMIT 5000;

GRANT SELECT ON ALL TABLES IN SCHEMA decl_sharing_demo.curated_data TO ROLE share_admin;

## 3. Create Secure Views

Secure views hide the underlying SQL definition from consumers. This is **required** for sharing
views via declarative sharing — non-secure views are not supported in manifests.

In [ ]:
USE ROLE share_admin;

CREATE OR REPLACE SECURE VIEW decl_sharing_demo.curated_data.order_summary AS
SELECT
    o.o_orderkey,
    o.o_orderdate,
    o.o_orderstatus,
    o.o_totalprice,
    o.o_orderpriority,
    c.c_name       AS customer_name,
    c.c_mktsegment AS market_segment
FROM decl_sharing_demo.curated_data.orders o
JOIN decl_sharing_demo.curated_data.customers c
  ON o.o_custkey = c.c_custkey;

CREATE OR REPLACE SECURE VIEW decl_sharing_demo.curated_data.customer_overview AS
SELECT
    c.c_custkey,
    c.c_name,
    c.c_mktsegment,
    c.c_nationkey,
    COUNT(o.o_orderkey)  AS total_orders,
    SUM(o.o_totalprice)  AS total_spend
FROM decl_sharing_demo.curated_data.customers c
LEFT JOIN decl_sharing_demo.curated_data.orders o
  ON c.c_custkey = o.o_custkey
GROUP BY c.c_custkey, c.c_name, c.c_mktsegment, c.c_nationkey;

SELECT * FROM decl_sharing_demo.curated_data.order_summary LIMIT 5;

## 4. Create Application Package (TYPE=DATA)

With `TYPE=DATA`, the package uses the **declarative manifest workflow**:
- A `manifest.yml` declares shared objects (no setup script needed)
- A **LIVE version** is automatically created for iterative development
- You **BUILD → COMMIT → RELEASE** to publish immutable versions

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE APPLICATION PACKAGE IF NOT EXISTS demo_data_package
    TYPE = DATA
    COMMENT = 'Versioned order analytics data product';

## 5. Create Manifest and Upload to Package

The **manifest.yml** declares the shared content structure. For `TYPE=DATA` packages,
you upload files directly to the LIVE version using `PUT` to the `snow://` URL scheme.

### Manifest structure

```yaml
roles:
  - DATA_VIEWER:
      comment: "Read-only access to order analytics data"

shared_content:
  databases:
    - DECL_SHARING_DEMO:
        schemas:
          - CURATED_DATA:
              roles: [DATA_VIEWER]
              tables:
                - ORDERS:
                    roles: [DATA_VIEWER]
                - CUSTOMERS:
                    roles: [DATA_VIEWER]
              views:
                - ORDER_SUMMARY:
                    roles: [DATA_VIEWER]
                - CUSTOMER_OVERVIEW:
                    roles: [DATA_VIEWER]
```

### Upload steps

Save the manifest locally and upload with:
```sql
PUT file:///path/to/manifest.yml
    snow://package/DEMO_DATA_PACKAGE/versions/LIVE/
    OVERWRITE=TRUE AUTO_COMPRESS=false;
```

Or from a Snowflake workspace:
```sql
COPY FILES INTO snow://package/DEMO_DATA_PACKAGE/versions/LIVE/
  FROM 'snow://workspace/<workspace>/versions/live/'
  FILES = ('manifest.yml');
```

**Note:** The `PUT`/`COPY FILES` commands above must be executed from the Snowflake CLI or Snowsight.
They are shown here for reference — the SQL cell below demonstrates verifying stage contents.

In [ ]:
-- List files in the LIVE version of the package
-- (will be empty until manifest.yml is uploaded via CLI/Snowsight)
LIST snow://package/DEMO_DATA_PACKAGE/versions/LIVE;

## 6. Grant Data to Application Package

For `TYPE=DATA` packages, use `GRANT ... TO SHARE IN APPLICATION PACKAGE` to make
database objects available to the package's shared content.

In [ ]:
GRANT USAGE ON DATABASE decl_sharing_demo
    TO SHARE IN APPLICATION PACKAGE demo_data_package;
GRANT USAGE ON SCHEMA decl_sharing_demo.curated_data
    TO SHARE IN APPLICATION PACKAGE demo_data_package;

GRANT SELECT ON TABLE decl_sharing_demo.curated_data.orders
    TO SHARE IN APPLICATION PACKAGE demo_data_package;
GRANT SELECT ON TABLE decl_sharing_demo.curated_data.customers
    TO SHARE IN APPLICATION PACKAGE demo_data_package;
GRANT SELECT ON VIEW decl_sharing_demo.curated_data.order_summary
    TO SHARE IN APPLICATION PACKAGE demo_data_package;
GRANT SELECT ON VIEW decl_sharing_demo.curated_data.customer_overview
    TO SHARE IN APPLICATION PACKAGE demo_data_package;

## 7. Build, Commit, and Release

The declarative workflow uses **LIVE versions** instead of numbered versions:

1. **BUILD** — validates the manifest and shared object references
2. **COMMIT** — creates an immutable snapshot from the LIVE version
3. **RELEASE** — makes the committed version available to consumers

Or use `RELEASE LIVE VERSION` to do all three in one step.

> **Legacy comparison:** The traditional Native App Framework uses `ADD VERSION v1_0 USING '@stage'`
> and `SET DEFAULT RELEASE DIRECTIVE VERSION = v1_0 PATCH = 0`. Those commands apply to
> `TYPE=STANDARD` packages with setup scripts — not `TYPE=DATA`.

In [ ]:
-- Declarative workflow (TYPE=DATA):
-- Step 1: Build (validates manifest)
-- ALTER APPLICATION PACKAGE demo_data_package BUILD;

-- Step 2: Commit (creates immutable version)
-- ALTER APPLICATION PACKAGE demo_data_package COMMIT;

-- Step 3: Release (publishes to consumers)
-- ALTER APPLICATION PACKAGE demo_data_package RELEASE;

-- Or all-in-one:
-- ALTER APPLICATION PACKAGE demo_data_package RELEASE LIVE VERSION;

-- Legacy versioned approach (TYPE=STANDARD with setup scripts):
-- ALTER APPLICATION PACKAGE demo_data_package
--     ADD VERSION v1_0 USING '@decl_sharing_demo.app_stage.v1_files';
-- ALTER APPLICATION PACKAGE demo_data_package
--     SET DEFAULT RELEASE DIRECTIVE VERSION = v1_0 PATCH = 0;

-- Verify the package
SHOW APPLICATION PACKAGES LIKE 'DEMO_DATA_PACKAGE';
DESCRIBE APPLICATION PACKAGE demo_data_package;

## 8. Distribution and Listing

To share this package with consumers in other accounts:

1. **Set distribution to EXTERNAL** — allows the package to be shared outside your organization
2. **Create a listing** — either a private listing targeting specific accounts, or a public Marketplace listing

```sql
-- Enable external distribution
ALTER APPLICATION PACKAGE demo_data_package SET DISTRIBUTION = EXTERNAL;

-- Create a private listing (replace with real consumer account)
CREATE EXTERNAL LISTING demo_data_listing
APPLICATION PACKAGE demo_data_package AS
$$
title: "Order Analytics Data Product"
subtitle: "Curated TPC-H order and customer data"
description: "Tables, secure views, and analytics-ready data."
listing_terms:
  type: "OFFLINE"
targets:
  accounts: ["CONSUMER_ORG.CONSUMER_ACCOUNT"]
$$
PUBLISH = FALSE
REVIEW = FALSE;

-- Publish when ready
ALTER LISTING demo_data_listing PUBLISH;
```

For **cross-region** sharing, declarative packages require:
```
auto_fulfillment.refresh_type: SUB_DATABASE_WITH_REFERENCE_USAGE
```

## 9. Verify Package Contents

In [ ]:
SHOW APPLICATION PACKAGES;
SHOW GRANTS TO SHARE IN APPLICATION PACKAGE demo_data_package;

## 10. Teardown

**Important:** Drop the listing first (if created), then the package, then the database.
You cannot drop a package while a published listing still references it.

In [ ]:
USE ROLE ACCOUNTADMIN;
-- DROP LISTING IF EXISTS demo_data_listing;
DROP APPLICATION PACKAGE IF EXISTS demo_data_package;
DROP DATABASE IF EXISTS decl_sharing_demo;
DROP ROLE IF EXISTS share_admin;
DROP ROLE IF EXISTS share_monitor;

## References

- [About Declarative Sharing](https://docs.snowflake.com/en/developer-guide/declarative-sharing/about)
- [Application Packages in Declarative Sharing](https://docs.snowflake.com/en/developer-guide/declarative-sharing/package)
- [Declarative Native App Manifest Reference](https://docs.snowflake.com/en/developer-guide/declarative-sharing/manifest-reference)
- [Package Versions in Declarative Sharing](https://docs.snowflake.com/en/developer-guide/declarative-sharing/versioning)
- [CREATE APPLICATION PACKAGE](https://docs.snowflake.com/en/sql-reference/sql/create-application-package)
- [Share Data Content in a Native App](https://docs.snowflake.com/en/developer-guide/native-apps/preparing-data-content)
- [Set Release Directives (Legacy)](https://docs.snowflake.com/en/developer-guide/native-apps/update-app-release-directive)
- [Creating a Listing](https://docs.snowflake.com/en/developer-guide/native-apps/creating-listing)